In [ ]:
# ======================================================================
# PROJECT 1 - Predicting crime rates with a neural network
# AI Builders Lab  ·  Class 2  ·  23 August
#
# CODE ONLY. Every cell, no explanations.
# The explained version, with diagrams, is:  01_Crime_Prediction_ANN.ipynb
#
# Run cells with Shift + Enter, in order, top to bottom.
# ======================================================================

# Run this cell first  (Shift + Enter)

import pandas as pd                                    # tables and spreadsheets
import numpy as np                                     # arrays of numbers
import matplotlib.pyplot as plt                        # graphs

import tensorflow as tf
from tensorflow.keras.models import Sequential         # "layers stacked in a row"
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam           # Adam = "adaptive moment estimation"
from sklearn.model_selection import train_test_split

# Fix the random seed. Neural networks start from random numbers, so without this
# your result and your neighbour's would differ every single run.
tf.keras.utils.set_random_seed(42)

print("TensorFlow version:", tf.__version__)
print("Toolbox is open.")

In [ ]:
# The data lives in our class GitHub repository, so pandas can read it straight off the web.
# No downloading, no Google Drive, no uploading. One line.

DATA_URL = "https://raw.githubusercontent.com/chizelnut/ABL_lab1/main/data/crimeSTATS.csv"

crime_data = pd.read_csv(DATA_URL, sep=',')

display(crime_data)          # in Colab, display() renders a table nicely


In [ ]:
# How big is this thing, and what is in it?

print("Shape (rows, columns):", crime_data.shape)
print("  ->", crime_data.shape[0], "SAMPLES (cities), described by", crime_data.shape[1], "columns")

target = 'total_crime_reported_per_1_million_res'

print("\nThe TARGET - the thing we want to predict (Y):")
print("   ", target)

print("\nThe FEATURES - the facts we get to look at (X):")
for i, name in enumerate(crime_data.columns.drop(target)):
    print("   ", i, name)

print("\nThe first 3 rows - these are SAMPLES, three real cities:")
display(crime_data.head(3))

In [ ]:
# Question 1, in one line: are there any blanks anywhere in the table?
print("Missing values in the whole table:", crime_data.isnull().sum().sum())

# Question 2: are there any IMPOSSIBLE values?
# A city cannot have a NEGATIVE number of crimes. Let's look.
print("\nLowest crime value in the data:", crime_data[target].min())
print("\nRows where crime is negative — these cannot be real:")
display(crime_data[crime_data[target] < 0])

In [ ]:
rows_before = len(crime_data)

crime_data = crime_data[crime_data[target] >= 0]

print("Rows before cleaning:", rows_before)
print("Rows after cleaning: ", len(crime_data))
print("Removed:", rows_before - len(crime_data), "impossible rows")

In [ ]:
X = crime_data.drop([target], axis=1).values     # everything EXCEPT the target -> the inputs
Y = crime_data[[target]].values                  # ONLY the target -> the answer

print("X shape:", X.shape, " <- 596 cities, 5 features each")
print("Y shape:", Y.shape, " <- 596 answers")

print("\nCity number 0 looks like this to the model:")
print("  X =", X[0], "  -> Y =", Y[0][0])

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.20,        # 20% held back for the exam
    random_state=25        # a fixed random seed, so the split is reproducible
)

print("Training on:", X_train.shape[0], "cities")
print("Testing on: ", X_test.shape[0], "cities  (the model will not see these until the end)")

In [ ]:
def create_model(learning_rate, dropout_rate):

    model = Sequential()                       # Sequential = layers in a straight line, one after another

    model.add(Input(shape=(X_train.shape[1],)))       # 5 features in

    model.add(Dense(100, activation='relu'))          # why ReLU? no upper bound for crime incidents
    model.add(Dropout(dropout_rate))                  # randomly zero some weights, to avoid overfitting

    model.add(Dense(50,  activation='relu'))
    model.add(Dropout(dropout_rate))

    model.add(Dense(25,  activation='relu'))
    model.add(Dropout(dropout_rate))

    model.add(Dense(1))                               # one number out. No activation - we want it unbounded.

    adam = Adam(learning_rate=learning_rate)
    model.compile(loss='mean_squared_error',          # MSE: the standard cost function for regression
                  optimizer=adam,
                  metrics=['mae'])                    # MAE: "on average, how far off are we?" - in real units
    return model


# The settings WE choose. These are called hyper-parameters:
# the model does not learn them, a human picks them.
dropout_rate  = 0.1
learn_rate    = 0.01
epochs        = 60

model = create_model(learn_rate, dropout_rate)
model.summary()

In [ ]:
model_history = model.fit(
    X_train, Y_train,
    batch_size=16,
    epochs=epochs,
    validation_split=0.2,
    verbose=1
)

print("\nTraining finished. The model now holds", model.count_params(), "learned numbers.")

In [ ]:
score = model.evaluate(X_test, Y_test, verbose=0)

print("Loss (MSE):          ", round(score[0], 1))
print("Mean Absolute Error: ", round(score[1], 1), "crimes per million residents")

# Is that good? Compare against the laziest possible model: always guess the average.
naive = float(np.abs(Y_test - Y_train.mean()).mean())
print("\nA model that just guesses the average every time would be off by:", round(naive, 1))
print("So our network is roughly", round(naive / score[1], 1), "times better than guessing.")

In [ ]:
# Graph the error for the training set and the validation set

plt.figure(figsize=(9, 5))
plt.plot(model_history.history['mae'], label='training data')
plt.plot(model_history.history['val_mae'], label='validation data (held out)')
plt.legend(loc='upper right')
plt.title('Model Error over Training')
plt.ylabel('Mean Absolute Error')
plt.xlabel('Epoch')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# A 2-D array, because the model expects a BATCH of samples - even a batch of one.
X_new = np.array([[55, 68, 14, 26, 17]], dtype=X_train.dtype)

prediction = model.predict(X_new, verbose=0)

print("Feature order the model expects:")
print("  ", list(crime_data.drop([target], axis=1).columns))
print("\nOur made-up city:", X_new[0])
print("\nPREDICTED total crime per 1 million residents:", round(float(prediction[0, 0]), 1))

In [ ]:
# Now change something and see what the model believes.
# Here: the same city, but with police funding raised from 55 to 80.

X_new_2 = np.array([[80, 68, 14, 26, 17]], dtype=X_train.dtype)
print("With police funding at 55:", round(float(model.predict(X_new,   verbose=0)[0,0]), 1))
print("With police funding at 80:", round(float(model.predict(X_new_2, verbose=0)[0,0]), 1))

In [ ]:
model.save('crime_model.keras')
print("Saved to crime_model.keras")

# Click the FOLDER icon in the left sidebar of Colab to see it.
# Colab deletes this storage when your session ends, so right-click -> Download to keep it.

# To load it back later, in any notebook:
# from tensorflow import keras
# model = keras.models.load_model('crime_model.keras')